# CALM / AXIOM++ Common-GPU Fuzzer (T4-ready, eager-vs-compiled)

Run-All notebook for defensive PyTorch API fuzzing. This is automated **differential testing** of a numerical library: run the same computation along several semantics-preserving paths and report only divergences that a stable build should not produce. The goal is not a large raw crash count; it is software-layer failures that reproduce across GPUs.

Surfaces, in scheduling priority:

1. **Proven cross-GPU repros** — backend abort / native crash / Windows access violation; `INTERNAL ASSERT FAILED` / `please report a bug`; CUDA device-side assert where CPU validates cleanly; silent sparse invariant gaps.
2. **Eager ↔ compiled equivalence differential (the surface that needs a real accelerator + Inductor).** The two "backends" are eager execution and `torch.compile(backend="inductor")` of the *same* function, in the same dtype and device. A conservative oracle fires only on finiteness-class divergence (compiled folds a NaN away, or invents one), shape/dtype/arity mismatch, large numeric divergence, or one path raising while the other returns a value. This is the equivalence-transform oracle that crash-only fuzzers lack, and it is exactly what local Windows could not run (no Triton / no C compiler).
3. Sparse / indexing / storage boundary contracts, then a randomized linalg battery with a classifier that filters clean `LinAlgError` / `IndexError` as `expected_raise` negative controls.

The compile probes target reported Inductor silent-correctness families (multiple `amax`/`amin` on one tensor, transpose fusion, LayerNorm/GroupNorm numerics, `addcdiv(x,x,clamp(x))`, `index_add_`, large-dim softmax) **and** structural patterns where a code generator is most likely to disagree with eager (`x - x` on NaN, `where()` with a NaN dead branch, `std`/`var` of a constant, `inf - inf` reductions, reassociated `exp`/`log`). The duplicate screen labels each hit as likely-duplicate / known-related / not-seen, with issue references, so nothing is passed off as novel without retest.

Two Gemma 4 SFT exports are produced: findings rows (what actually failed) and a **compile-diff seed catalog** (`calm_common_gpu_gemma4_seeds.jsonl`) that teaches the model to *generate* eager-vs-compiled differential probes, independent of whether this build diverged.


In [ ]:
import os, sys, json, platform, subprocess, zipfile
from pathlib import Path

try:
    import torch
    print("torch", torch.__version__)
    print("cuda", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
        cap = torch.cuda.get_device_capability(0)
        print("note: sm_75 (T4) has no TF32; the value on T4 is the compile/Inductor surface, not hardware precision")
except Exception as e:
    print("torch import failed:", type(e).__name__, e)

ROOT = Path.cwd()
print("workdir", ROOT)


## Knobs

`QUICK=True` is a short validation run. `QUICK=False` is the balanced T4 run. `COMPILE_BACKEND="inductor"` is the real hunt on Colab T4 / Blackwell; the preflight cell auto-falls back to `aot_eager` if Inductor is unavailable so Run-All never dies. `COMPILE_DEVICES="cuda"` matches T4's value (set `"cpu,cuda"` to also fuzz CPU codegen).


In [ ]:
QUICK = False
COMPILE_BACKEND = "inductor"   # real Inductor/Triton on T4; preflight auto-falls back if missing
COMPILE_DEVICES = "cuda"       # T4 value is cuda+Inductor; use "cpu,cuda" to also fuzz CPU codegen
FUZZ_SECONDS = 300 if QUICK else 1500
MAX_CHILD = 48 if QUICK else 140
CHILD_TIMEOUT = 25             # default per-probe; compile probes carry their own 90s budget
SEED = 20260611
print(dict(QUICK=QUICK, COMPILE_BACKEND=COMPILE_BACKEND, COMPILE_DEVICES=COMPILE_DEVICES,
           FUZZ_SECONDS=FUZZ_SECONDS, MAX_CHILD=MAX_CHILD, CHILD_TIMEOUT=CHILD_TIMEOUT, SEED=SEED))


In [ ]:
%%writefile calm_hunt_v2_common_gpu.py
#!/usr/bin/env python3
"""
CALM hunt v2: common-GPU PyTorch bug hunter.

Goal:
  - Reproduce software-layer bugs that are shared across GPUs, not only hardware
    precision effects.
  - Run dangerous crash probes in child Python processes.
  - Keep CPU-vs-CUDA exact differentials in-process and conservative.
  - Emit JSON + small repro scripts that can be copied into a report or issue.

This is a defensive QA tool for DL libraries. It does not use networking, files
outside its artifact directory, ctypes, shell payloads, or third-party services.
"""

from __future__ import annotations

import json
import os
import subprocess
import sys
import textwrap
import time
from pathlib import Path
from typing import Any, Callable

import torch

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

ROOT = Path(__file__).resolve().parent
REPRO_DIR = ROOT / "calm_hunt_v2_repros"
OUT_JSON = ROOT / "calm_hunt_v2_findings.json"
OUT_MD = ROOT / "calm_hunt_v2_summary.md"
PYTHON = sys.executable
CUDA = torch.cuda.is_available()
DEVICE = "cuda" if CUDA else "cpu"

INT_DTYPES = (torch.int8, torch.int16, torch.int32, torch.int64)


def tail(text: str, n: int = 4000) -> str:
    return text[-n:] if text and len(text) > n else (text or "")


def write_repro(name: str, body: str) -> Path:
    REPRO_DIR.mkdir(parents=True, exist_ok=True)
    path = REPRO_DIR / f"{name}.py"
    prelude = """
import os
os.environ.setdefault("PYTHONHASHSEED", "0")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
import torch
torch.manual_seed(0)
try:
    torch.set_num_threads(1)
except Exception:
    pass
"""
    path.write_text(textwrap.dedent(prelude + "\n" + body).strip() + "\n", encoding="utf-8")
    return path


def classify_child(returncode: int | None, stdout: str, stderr: str, timed_out: bool) -> tuple[str, str]:
    joined = (stdout or "") + "\n" + (stderr or "")
    low = joined.lower()
    if timed_out:
        return "timeout", "Tier A: child process timed out"
    if "skip:" in low and returncode == 0:
        return "skip", "Not applicable on this machine"
    if "silent_invalid_sparse" in low:
        return "contract_silent", "Tier B: invalid sparse index accepted and materialized"
    if returncode == 0:
        return "ok", "No failure observed"
    if "internal assert failed" in low or "please report a bug to pytorch" in low:
        return "internal_assert", "Tier A: PyTorch INTERNAL ASSERT"
    if "device-side assert triggered" in low or "illegal memory access" in low:
        return "cuda_assert", "Tier A: CUDA device-side assert"
    if "intel onemkl error" in low or "mkl error" in low:
        return "backend_abort", "Tier A: backend library abort/error path"
    if "segmentation fault" in low or "access violation" in low or "0xc0000005" in low:
        return "native_crash", "Tier A: native crash"
    if "runtimeerror" in low:
        return "runtime_error", "Tier B: runtime error / contract failure"
    return "hard_exit", "Tier A/B: non-zero process exit without a clean result"


def run_child_probe(name: str, body: str, timeout_sec: float = 25.0) -> dict[str, Any]:
    path = write_repro(name, body)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env.setdefault("CUDA_LAUNCH_BLOCKING", "1")
    started = time.time()
    try:
        proc = subprocess.run(
            [PYTHON, "-u", str(path)],
            cwd=str(ROOT),
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout_sec,
            env=env,
        )
        runtime = time.time() - started
        status, verdict = classify_child(proc.returncode, proc.stdout, proc.stderr, False)
        return {
            "name": name,
            "phase": "child_repro",
            "status": status,
            "verdict": verdict,
            "returncode": proc.returncode,
            "runtime_sec": round(runtime, 3),
            "repro": str(path),
            "stdout_tail": tail(proc.stdout),
            "stderr_tail": tail(proc.stderr),
        }
    except subprocess.TimeoutExpired as exc:
        status, verdict = classify_child(None, exc.stdout or "", exc.stderr or "", True)
        return {
            "name": name,
            "phase": "child_repro",
            "status": status,
            "verdict": verdict,
            "returncode": None,
            "runtime_sec": timeout_sec,
            "repro": str(path),
            "stdout_tail": tail(exc.stdout or ""),
            "stderr_tail": tail(exc.stderr or ""),
        }


def child_probes() -> list[tuple[str, str]]:
    return [
        (
            "eigvals_2x2_offdiag_nan_cpu",
            """
print("probe=eigvals offdiag NaN cpu", flush=True)
A = torch.tensor([[1.0, float("nan")], [float("nan"), 1.0]], device="cpu")
print(torch.linalg.eigvals(A), flush=True)
""",
        ),
        (
            "eigvals_2x2_offdiag_nan_cuda",
            """
if not torch.cuda.is_available():
    print("SKIP: no cuda", flush=True)
else:
    print("probe=eigvals offdiag NaN cuda", flush=True)
    A = torch.tensor([[1.0, float("nan")], [float("nan"), 1.0]], device="cuda")
    y = torch.linalg.eigvals(A)
    torch.cuda.synchronize()
    print(y, flush=True)
""",
        ),
        (
            "eigvals_2x2_offdiag_inf_cpu",
            """
print("probe=eigvals offdiag Inf cpu", flush=True)
A = torch.tensor([[1.0, float("inf")], [float("-inf"), 1.0]], device="cpu")
print(torch.linalg.eigvals(A), flush=True)
""",
        ),
        (
            "lstsq_nan_A_cpu",
            """
print("probe=lstsq NaN A cpu", flush=True)
A = torch.tensor([[1.0, float("nan")], [2.0, 3.0]], device="cpu")
B = torch.ones(2, 1)
print(torch.linalg.lstsq(A, B).solution, flush=True)
""",
        ),
        (
            "lstsq_nan_A_cuda",
            """
if not torch.cuda.is_available():
    print("SKIP: no cuda", flush=True)
else:
    print("probe=lstsq NaN A cuda", flush=True)
    A = torch.tensor([[1.0, float("nan")], [2.0, 3.0]], device="cuda")
    B = torch.ones(2, 1, device="cuda")
    y = torch.linalg.lstsq(A, B).solution
    torch.cuda.synchronize()
    print(y, flush=True)
""",
        ),
        (
            "sparse_coo_negative_index_cpu",
            """
print("probe=sparse COO negative index cpu", flush=True)
idx = torch.tensor([[-1, 0], [0, 1]])
val = torch.ones(2)
x = torch.sparse_coo_tensor(idx, val, (2, 2), check_invariants=False)
dense = x.to_dense()
print("dense=", dense, flush=True)
try:
    torch.sparse_coo_tensor(idx, val, (2, 2), check_invariants=True)
except Exception as e:
    print("check_invariants_true_raises=", type(e).__name__, str(e)[:120], flush=True)
print("SILENT_INVALID_SPARSE", flush=True)
""",
        ),
        (
            "sparse_coo_negative_index_cuda_to_dense",
            """
if not torch.cuda.is_available():
    print("SKIP: no cuda", flush=True)
else:
    print("probe=sparse COO negative index cuda to_dense", flush=True)
    idx = torch.tensor([[-1, 0], [0, 1]], device="cuda")
    val = torch.ones(2, device="cuda")
    x = torch.sparse_coo_tensor(idx, val, (2, 2), check_invariants=False)
    y = x.to_dense()
    torch.cuda.synchronize()
    print(y, flush=True)
""",
        ),
        (
            "storage_resize_zero_then_clone_cpu",
            """
print("probe=storage resize zero then clone cpu", flush=True)
x = torch.arange(4)
s = x.untyped_storage()
print("before", x, flush=True)
s.resize_(0)
print("after resize storage", flush=True)
print(x.clone(), flush=True)
""",
        ),
        (
            "storage_resize_zero_then_repr_cpu",
            """
print("probe=storage resize zero then repr cpu", flush=True)
x = torch.arange(4)
s = x.untyped_storage()
print("before", x, flush=True)
s.resize_(0)
print("after resize storage", flush=True)
print(repr(x), flush=True)
""",
        ),
        (
            "storage_resize_zero_then_clone_cuda",
            """
if not torch.cuda.is_available():
    print("SKIP: no cuda", flush=True)
else:
    print("probe=storage resize zero then clone cuda", flush=True)
    x = torch.arange(4, device="cuda")
    torch.cuda.synchronize()
    s = x.untyped_storage()
    print("before", x, flush=True)
    s.resize_(0)
    torch.cuda.synchronize()
    print("after resize storage", flush=True)
    y = x.clone()
    torch.cuda.synchronize()
    print(y, flush=True)
""",
        ),
    ]


def clone_to(x: Any, device: str) -> Any:
    if torch.is_tensor(x):
        return x.detach().clone().to(device)
    if isinstance(x, (tuple, list)):
        return type(x)(clone_to(v, device) for v in x)
    return x


def run_callable(fn: Callable[..., Any], args: list[Any]) -> Any:
    try:
        out = fn(*args)
        if CUDA and any(torch.is_tensor(x) and x.is_cuda for x in args):
            torch.cuda.synchronize()
        return out
    except Exception as exc:
        return exc


def flatten_result(x: Any) -> list[Any]:
    if isinstance(x, (tuple, list)):
        out: list[Any] = []
        for item in x:
            out.extend(flatten_result(item))
        return out
    return [x]


def diff_results(cpu_out: Any, cuda_out: Any) -> tuple[str | None, str]:
    if isinstance(cpu_out, Exception) or isinstance(cuda_out, Exception):
        if isinstance(cpu_out, Exception) and isinstance(cuda_out, Exception):
            ctype, gtype = type(cpu_out).__name__, type(cuda_out).__name__
            if ctype == gtype:
                return None, f"both raised {ctype}"
            return "exception_type", f"cpu={ctype}: {str(cpu_out)[:120]} | cuda={gtype}: {str(cuda_out)[:120]}"
        side = "cpu" if isinstance(cpu_out, Exception) else "cuda"
        exc = cpu_out if isinstance(cpu_out, Exception) else cuda_out
        return "exception_side", f"{side} only {type(exc).__name__}: {str(exc)[:180]}"

    cpu_items, cuda_items = flatten_result(cpu_out), flatten_result(cuda_out)
    if len(cpu_items) != len(cuda_items):
        return "arity", f"{len(cpu_items)} vs {len(cuda_items)}"
    for a, b in zip(cpu_items, cuda_items):
        if not torch.is_tensor(a) or not torch.is_tensor(b):
            continue
        aa, bb = a.detach().cpu(), b.detach().cpu()
        if aa.shape != bb.shape:
            return "shape", f"{tuple(aa.shape)} vs {tuple(bb.shape)}"
        if aa.dtype != bb.dtype:
            return "dtype", f"{aa.dtype} vs {bb.dtype}"
        if aa.dtype in INT_DTYPES or aa.dtype == torch.bool:
            delta = aa != bb
            n = int(delta.sum())
            if n:
                max_abs = int((aa.to(torch.int64) - bb.to(torch.int64)).abs().max())
                return "int_exact", f"{n}/{aa.numel()} elements differ; max_abs={max_abs}"
            continue
        af, bf = aa.double(), bb.double()
        finite_a, finite_b = torch.isfinite(af), torch.isfinite(bf)
        if not torch.equal(finite_a, finite_b):
            n = int((finite_a ^ finite_b).sum())
            return "finiteness", f"finite/nonfinite class differs at {n} elements"
        diff = torch.where(finite_a & finite_b, (af - bf).abs(), torch.zeros_like(af))
        denom = bf.abs().clamp_min(1e-12)
        rel = diff / denom
        max_abs, max_rel = float(diff.max()), float(rel.max())
        if max_abs > 1e-3 and max_rel > 1e-2:
            return "float_large", f"max_abs={max_abs:.4g}; max_rel={max_rel:.4g}"
    return None, "same"


def make_device_cases() -> list[tuple[str, Callable[..., Any], list[Any]]]:
    torch.manual_seed(1234)
    i32 = torch.randint(-1000, 1000, (257,), dtype=torch.int32)
    i64 = torch.randint(-100000, 100000, (17, 19), dtype=torch.int64)
    pos_i32 = torch.randint(1, 31, (257,), dtype=torch.int32)
    shift = torch.randint(0, 7, (257,), dtype=torch.int32)
    f32 = torch.randn(31, 37, dtype=torch.float32)
    f32_small = torch.randint(-4, 5, (64, 64), dtype=torch.int32).float()
    special = torch.tensor(
        [float("nan"), float("inf"), float("-inf"), 0.0, -0.0, 1.0, -1.0, 88.7, -88.7],
        dtype=torch.float32,
    )
    gather_src = torch.arange(8 * 16, dtype=torch.float32).reshape(8, 16)
    gather_idx = torch.randint(0, 16, (8, 16), dtype=torch.int64)
    cases: list[tuple[str, Callable[..., Any], list[Any]]] = [
        ("floor_divide_i32", lambda a, b: torch.floor_divide(a, b), [i32, pos_i32]),
        ("remainder_i32_neg_divisor", lambda a, b: torch.remainder(a, -b), [i32, pos_i32]),
        ("fmod_i32_neg_divisor", lambda a, b: torch.fmod(a, -b), [i32, pos_i32]),
        ("div_floor_i32", lambda a, b: torch.div(a, b, rounding_mode="floor"), [i32, pos_i32]),
        ("div_trunc_i32", lambda a, b: torch.div(a, b, rounding_mode="trunc"), [i32, pos_i32]),
        ("bitshift_right_i32", lambda a, b: a >> b, [i32, shift]),
        ("bitshift_left_i32", lambda a, b: a << b, [i32.clamp(-32, 32), shift]),
        ("sum_i64_dim", lambda a: torch.sum(a, dim=-1), [i64]),
        ("prod_i32_dim", lambda a: torch.prod(a.reshape(17, -1).clamp(-3, 3), dim=-1), [i32[:255]]),
        ("cumsum_i32", lambda a: torch.cumsum(a, dim=-1), [i32]),
        ("cumprod_i32_small", lambda a: torch.cumprod(a.reshape(17, -1).clamp(-2, 2), dim=-1), [i32[:255]]),
        ("where_mixed", lambda c, a, b: torch.where(c, a, b), [i32[:128] > 0, i32[:128].float(), i64.flatten()[:128].float()]),
        ("gather_f32", lambda a, idx: torch.gather(a, -1, idx), [gather_src, gather_idx]),
        ("sort_i32_values", lambda a: torch.sort(a).values, [i32]),
        ("topk_i32_values", lambda a: torch.topk(a, k=31).values, [i32]),
        ("logsumexp_big", lambda a: torch.logsumexp(a * 100.0, dim=-1), [f32]),
        ("softmax_smallint", lambda a: torch.softmax(a, dim=-1), [f32_small]),
        ("special_exp", lambda a: torch.exp(a), [special]),
        ("special_log1p", lambda a: torch.log1p(a), [special]),
        ("special_erfc", lambda a: torch.erfc(a), [special]),
        ("noncontig_add", lambda a: a.t() + a, [f32_small]),
    ]
    return cases


def run_device_differential() -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    if not CUDA:
        return rows
    for name, fn, args in make_device_cases():
        cpu_args = [clone_to(x, "cpu") for x in args]
        cuda_args = [clone_to(x, "cuda") for x in args]
        cpu_out = run_callable(fn, cpu_args)
        cuda_out = run_callable(fn, cuda_args)
        kind, detail = diff_results(cpu_out, cuda_out)
        if kind:
            rows.append({
                "name": name,
                "phase": "cpu_cuda_differential",
                "status": kind,
                "verdict": "Tier A/B: CPU and CUDA diverged under conservative comparator",
                "detail": detail,
            })
    return rows


def compile_preflight() -> dict[str, Any]:
    row = {"available": False, "detail": ""}
    try:
        dev = "cuda" if CUDA else "cpu"
        fn = torch.compile(lambda x: x + 1, fullgraph=True, dynamic=False)
        y = fn(torch.ones(8, device=dev))
        if CUDA:
            torch.cuda.synchronize()
        row["available"] = bool(torch.allclose(y.cpu(), torch.ones(8) + 1))
        row["detail"] = "simple compile path succeeded"
    except Exception as exc:
        row["detail"] = f"{type(exc).__name__}: {str(exc)[:300]}"
    return row


def is_actionable(row: dict[str, Any]) -> bool:
    return row.get("status") not in {"ok", "skip", None}


def write_markdown(payload: dict[str, Any]) -> None:
    actionable = [r for r in payload["results"] if is_actionable(r)]
    tier_a = [r for r in actionable if str(r.get("verdict", "")).startswith("Tier A")]
    tier_b = [r for r in actionable if "Tier B" in str(r.get("verdict", ""))]
    lines = [
        "# CALM hunt v2 summary",
        "",
        f"- torch: `{payload['torch']}`",
        f"- cuda: `{payload['cuda']}`",
        f"- gpu: `{payload.get('gpu_name') or 'none'}`",
        f"- compile preflight: `{payload['compile_preflight']['detail']}`",
        f"- actionable findings: **{len(actionable)}** ({len(tier_a)} Tier A, {len(tier_b)} Tier B/mixed)",
        "",
        "## Findings",
        "",
    ]
    for r in actionable:
        lines.append(f"- **{r['name']}**: `{r['status']}` - {r.get('verdict', '')}")
        if r.get("detail"):
            lines.append(f"  - detail: `{r['detail']}`")
        if r.get("repro"):
            lines.append(f"  - repro: `{r['repro']}`")
        out = (r.get("stdout_tail") or "").strip().splitlines()
        err = (r.get("stderr_tail") or "").strip().splitlines()
        sample = (out + err)[:4]
        if sample:
            lines.append("  - signal: `" + " | ".join(s[:160] for s in sample) + "`")
    lines.append("")
    OUT_MD.write_text("\n".join(lines), encoding="utf-8")


def main() -> int:
    print("=" * 80)
    print(f"CALM hunt v2 common-GPU | torch={torch.__version__} | cuda={CUDA}")
    if CUDA:
        print(f"GPU={torch.cuda.get_device_name(0)} cap={torch.cuda.get_device_capability(0)}")
    print("=" * 80)

    results: list[dict[str, Any]] = []
    print("[1/3] child crash/contract probes")
    for name, body in child_probes():
        row = run_child_probe(name, body)
        results.append(row)
        mark = "HIT" if is_actionable(row) else row["status"].upper()
        print(f"  {mark:4s} {name:42s} {row['status']:18s} rc={row['returncode']}")

    print("[2/3] CPU-vs-CUDA exact differential probes")
    dd = run_device_differential()
    results.extend(dd)
    if dd:
        for row in dd:
            print(f"  HIT  {row['name']:42s} {row['status']:18s} {row.get('detail', '')}")
    else:
        print("  no conservative CPU/CUDA divergence found")

    print("[3/3] torch.compile preflight")
    cpre = compile_preflight()
    print("  " + cpre["detail"])

    payload = {
        "tool": "calm_hunt_v2_common_gpu",
        "torch": torch.__version__,
        "cuda": CUDA,
        "gpu_name": torch.cuda.get_device_name(0) if CUDA else None,
        "gpu_capability": torch.cuda.get_device_capability(0) if CUDA else None,
        "compile_preflight": cpre,
        "results": results,
        "summary": {
            "total": len(results),
            "actionable": sum(1 for r in results if is_actionable(r)),
            "by_status": {},
        },
    }
    by_status: dict[str, int] = {}
    for row in results:
        by_status[row["status"]] = by_status.get(row["status"], 0) + 1
    payload["summary"]["by_status"] = by_status
    OUT_JSON.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    write_markdown(payload)
    print("-" * 80)
    print(json.dumps(payload["summary"], ensure_ascii=False, indent=2))
    print(f"-> {OUT_JSON}")
    print(f"-> {OUT_MD}")
    print(f"-> repros in {REPRO_DIR}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile calm_compile_diff_probes.py
#!/usr/bin/env python3
"""
CALM compile-differential probes (eager vs torch.compile).

This is the surface that local Windows could not reach (no Triton / no C
compiler), and the one that actually needs a real accelerator + Inductor:
Google Colab T4, or Blackwell. It implements an EQUIVALENCE-TRANSFORM oracle.
The two "backends" being compared are two semantics-preserving execution paths
of the *same* computation:

    eager execution   vs   torch.compile(..., backend="inductor")

Both run in the same dtype and on the same device, so a stable PyTorch build
should agree up to benign floating-point reordering. The comparator (reused
from calm_hunt_v2_common_gpu.diff_results) only fires on:

    * finiteness-class divergence  (compiled returns NaN/Inf where eager is
      finite, or folds a NaN away)  <- the cleanest silent-correctness signal
    * shape / dtype / arity mismatch
    * large numeric divergence (max_abs > 1e-3 AND max_rel > 1e-2)
    * one path raises while the other returns a value

It deliberately does NOT count:

    * compiled raising BackendCompilerFailed / Unsupported while eager works
      -> COMPILE_UNSUPPORTED, logged as a negative control (this is a compile
      *limitation*, not a numerical bug);
    * benign exception-type re-wrapping under compile -> COMPILE_EXC_TYPE.

Why this catalog: the structural probes (x - x on NaN, where() with a NaN dead
branch, std() of a constant tensor, inf - inf reductions, rsqrt(0), reassociated
exp/log) target the constant-folding / reassociation / reduction-fusion classes
where a code generator is most likely to silently disagree with eager. The named
probes reproduce reported Inductor silent-correctness families so the fuzzer's
duplicate screen can label them honestly instead of pretending they are novel.

Each probe runs in a child process. A codegen segfault, CUDA illegal memory
access, or INTERNAL ASSERT under compile is therefore caught by the subprocess
classifier as a native crash / device assert, not a kernel death.

Env knobs (read inside the child):
    CALM_COMPILE_BACKEND   default "inductor"; set "aot_eager" to self-test the
                           harness on a path that needs no Triton / C compiler.
"""

from __future__ import annotations

from typing import Any

# Per-probe child timeout: a cold inductor compile (Triton autotune on first
# call in a fresh process) can take tens of seconds; eager/aot_eager is fast.
COMPILE_TIMEOUT = 90.0

# (name, fn_src, inp_src, devices, dynamic, fullgraph)
# fn_src must define a top-level callable `f`.
# inp_src must define a top-level tuple `inputs`, may reference `dev`.
_CASES: list[tuple[str, str, str, tuple[str, ...], bool, bool]] = [
    # ---- structural / finiteness: semantics-preserving, eager MUST match ----
    (
        "sub_self_nan",
        "def f(x):\n    return x - x",
        "inputs = (torch.tensor([float('nan'), float('inf'), float('-inf'), 1.0, -2.0, 0.0, 1e30, -1e30], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "mul_zero_special",
        "def f(x):\n    return x * 0.0",
        "inputs = (torch.tensor([float('nan'), float('inf'), float('-inf'), 1.0, -2.0, 0.0, 1e30, -1e30], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "where_nan_dead_branch",
        "def f(a, b, m):\n    return torch.where(m, a, b) + 0.0",
        "a = torch.arange(8, device=dev, dtype=torch.float32)\n"
        "b = torch.full((8,), float('nan'), device=dev)\n"
        "m = torch.ones(8, dtype=torch.bool, device=dev)\n"
        "inputs = (a, b, m)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "std_constant",
        "def f(x):\n    return torch.std(x, dim=-1)",
        "inputs = (torch.full((4, 64), 3.141592653589793, device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "var_constant",
        "def f(x):\n    return torch.var(x, dim=-1)",
        "inputs = (torch.full((4, 64), -2.718281828, device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "sum_inf_cancel",
        "def f(x):\n    return x.sum(dim=-1)",
        "inputs = (torch.tensor([[1e38, 1e38, -1e38, -1e38, 1.0]], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "rsqrt_zero",
        "def f(x):\n    return torch.rsqrt(x)",
        "inputs = (torch.tensor([0.0, 1.0, 4.0, 1e-30, float('inf')], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "reciprocal_zero",
        "def f(x):\n    return torch.reciprocal(x)",
        "inputs = (torch.tensor([0.0, -0.0, 1.0, 1e-300, float('inf'), float('nan')], dtype=torch.float64, device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "exp_log_reassoc",
        "def f(x):\n    return torch.log(torch.exp(x))",
        "inputs = (torch.tensor([0.0, 1.0, 80.0, 100.0, 700.0, -700.0], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "nan_to_num_fuse",
        "def f(x):\n    return torch.nan_to_num(x / x, nan=0.0, posinf=1.0, neginf=-1.0)",
        "inputs = (torch.tensor([float('nan'), 0.0, float('inf'), -3.0, 2.0], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "clamp_special_bounds",
        "def f(x):\n    return torch.clamp(x, min=0.0, max=1.0)",
        "inputs = (torch.tensor([float('nan'), float('inf'), float('-inf'), -1.0, 0.5, 2.0], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "gelu_extreme",
        "def f(x):\n    return torch.nn.functional.gelu(x)",
        "inputs = (torch.tensor([-1e4, -50.0, -5.0, 0.0, 5.0, 50.0, 1e4], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "silu_extreme",
        "def f(x):\n    return torch.nn.functional.silu(x)",
        "inputs = (torch.tensor([-1e4, -50.0, -5.0, 0.0, 5.0, 50.0, 1e4], device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    # ---- reported Inductor silent-correctness families (dup-screened) ----
    (
        "multi_amax_amin_same",  # issue #179577 reuse_partial
        "def f(x):\n    a = torch.amax(x, dim=2)\n    b = torch.amin(x, dim=1)\n    c = torch.amax(x, dim=(1, 2))\n    return a.sum() + b.sum() + c.sum()",
        "inputs = (torch.randn(4, 8, 8, device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "transpose_scale_fuse",  # issue #146416 transpose fusion
        "def f(x, s):\n    return (x.transpose(-1, -2) * s).contiguous()",
        "x = torch.randn(16, 32, device=dev)\ns = torch.randn(32, 1, device=dev)\ninputs = (x, s)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "layernorm_numerical",  # issue #143182 norm numerics
        "def f(x, w, b):\n    return torch.nn.functional.layer_norm(x, (64,), w, b, 1e-5)",
        "x = torch.randn(8, 64, device=dev) * 1e3\nw = torch.randn(64, device=dev)\nb = torch.randn(64, device=dev)\ninputs = (x, w, b)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "addcdiv_xx_clamp",  # issue #181694 exact pattern
        "def f(x):\n    return torch.addcdiv(x, x, torch.clamp(x, min=1e-6))",
        "inputs = (torch.randn(256, device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "index_add_accum",  # issue #111133 index_add mismatch
        "def f(x, idx, src):\n    y = x.clone()\n    return y.index_add_(0, idx, src)",
        "x = torch.zeros(5, 4, device=dev)\nidx = torch.tensor([0, 1, 0, 2, 1, 0], device=dev)\nsrc = torch.randn(6, 4, device=dev)\ninputs = (x, idx, src)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "softmax_bigdim_split",  # issue #157975 online-softmax reduction split
        "def f(x):\n    return torch.softmax(x, dim=-1)",
        "inputs = (torch.randn(2, 8192, device=dev) * 10.0,)",
        ("cpu", "cuda"), False, True,
    ),
    # ---- general op-fusion under extreme inputs ----
    (
        "masked_softmax_attn",
        "def f(s, m):\n    s = s.masked_fill(m, float('-inf'))\n    return torch.softmax(s, dim=-1)",
        "s = torch.randn(4, 16, device=dev)\nm = torch.zeros(4, 16, dtype=torch.bool, device=dev)\nm[:, 8:] = True\ninputs = (s, m)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "addmm_relu_large",
        "def f(a, b, c):\n    return torch.relu(torch.addmm(c, a, b))",
        "a = torch.randn(32, 48, device=dev) * 30.0\nb = torch.randn(48, 24, device=dev) * 30.0\nc = torch.randn(24, device=dev) * 30.0\ninputs = (a, b, c)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "matmul_layernorm",
        "def f(x, w):\n    y = x @ w\n    return torch.nn.functional.layer_norm(y, (32,))",
        "x = torch.randn(8, 32, device=dev)\nw = torch.randn(32, 32, device=dev)\ninputs = (x, w)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "groupnorm_numeric",
        "def f(x, w, b):\n    return torch.nn.functional.group_norm(x, 4, w, b)",
        "x = torch.randn(2, 8, 4, 4, device=dev) * 1e2\nw = torch.randn(8, device=dev)\nb = torch.randn(8, device=dev)\ninputs = (x, w, b)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "cumsum_long",
        "def f(x):\n    return torch.cumsum(x, dim=-1)",
        "inputs = (torch.randn(2, 4096, device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "logsumexp_big",
        "def f(x):\n    return torch.logsumexp(x * 50.0, dim=-1)",
        "inputs = (torch.randn(8, 128, device=dev),)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "scatter_overwrite",
        "def f(x, idx, src):\n    y = x.clone()\n    return y.scatter_(1, idx, src)",
        "x = torch.zeros(4, 8, device=dev)\nidx = torch.arange(8, device=dev).remainder(8).unsqueeze(0).expand(4, 8).contiguous()\nsrc = torch.randn(4, 8, device=dev)\ninputs = (x, idx, src)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "addcmul_chain",
        "def f(a, b, c):\n    return torch.addcmul(a, b, c, value=2.0)",
        "a = torch.randn(128, device=dev)\nb = torch.randn(128, device=dev)\nc = torch.randn(128, device=dev)\ninputs = (a, b, c)",
        ("cpu", "cuda"), False, True,
    ),
    (
        "bitshift_int_fuse",
        "def f(a, b):\n    return (a << b) >> 1",
        "a = torch.randint(-2**30, 2**30, (256,), dtype=torch.int32, device=dev)\nb = torch.randint(0, 6, (256,), dtype=torch.int32, device=dev)\ninputs = (a, b)",
        ("cpu", "cuda"), False, True,
    ),
    # ---- dynamic-shape codegen path (distinct from static) ----
    (
        "softmax_bigdim_dyn",
        "def f(x):\n    return torch.softmax(x, dim=-1)",
        "inputs = (torch.randn(3, 6000, device=dev) * 10.0,)",
        ("cpu", "cuda"), True, True,
    ),
    (
        "layernorm_dyn",
        "def f(x, w, b):\n    return torch.nn.functional.layer_norm(x, (48,), w, b, 1e-5)",
        "x = torch.randn(10, 48, device=dev) * 1e3\nw = torch.randn(48, device=dev)\nb = torch.randn(48, device=dev)\ninputs = (x, w, b)",
        ("cpu", "cuda"), True, True,
    ),
    (
        "cumsum_long_dyn",
        "def f(x):\n    return torch.cumsum(x, dim=-1)",
        "inputs = (torch.randn(5, 3000, device=dev),)",
        ("cpu", "cuda"), True, True,
    ),
]


# Self-contained: the child script only needs torch (it does NOT import any
# project module), so every emitted repro is standalone and shareable, and is a
# clean training example for the Gemma 4 SFT export.
_EPILOGUE = r'''
import os as _os
backend = _os.environ.get("CALM_COMPILE_BACKEND", "inductor")
try:
    import torch._dynamo as _dynamo
    import torch._dynamo.exc as _de
    _dynamo.reset()
    _dynamo.config.recompile_limit = 64
    _INFRA = tuple(t for t in (getattr(_de, "BackendCompilerFailed", None), getattr(_de, "Unsupported", None)) if isinstance(t, type))
except Exception:
    _INFRA = ()

_INT = (torch.int8, torch.int16, torch.int32, torch.int64)


def _is_infra(x):
    if not isinstance(x, Exception):
        return False
    if _INFRA and isinstance(x, _INFRA):
        return True
    nm = (type(x).__name__ + " " + str(x)[:200]).lower()
    return any(t in nm for t in ("backendcompiler", "unsupported", "cannot find a working triton", "compiler: cl is not found", "no module named 'triton'", "no module named 'filelock'"))


def _flat(x):
    if isinstance(x, (tuple, list)):
        o = []
        for v in x:
            o += _flat(v)
        return o
    return [x]


def _run(fn):
    try:
        out = fn(*inputs)
        if torch.cuda.is_available() and any(torch.is_tensor(t) and t.is_cuda for t in inputs):
            torch.cuda.synchronize()
        return out
    except Exception as e:
        return e


def _diff(a, b):
    if isinstance(a, Exception) or isinstance(b, Exception):
        if isinstance(a, Exception) and isinstance(b, Exception):
            if type(a).__name__ == type(b).__name__:
                return (None, "both raised " + type(a).__name__)
            return ("exception_type", type(a).__name__ + " vs " + type(b).__name__)
        side = "eager" if isinstance(a, Exception) else "compiled"
        e = a if isinstance(a, Exception) else b
        return ("exception_side", side + " only " + type(e).__name__ + ": " + str(e)[:160])
    ai, bi = _flat(a), _flat(b)
    if len(ai) != len(bi):
        return ("arity", str(len(ai)) + " vs " + str(len(bi)))
    for x, y in zip(ai, bi):
        if not torch.is_tensor(x) or not torch.is_tensor(y):
            continue
        x = x.detach().cpu()
        y = y.detach().cpu()
        if x.shape != y.shape:
            return ("shape", str(tuple(x.shape)) + " vs " + str(tuple(y.shape)))
        if x.dtype != y.dtype:
            return ("dtype", str(x.dtype) + " vs " + str(y.dtype))
        if x.dtype in _INT or x.dtype == torch.bool:
            d = (x != y)
            n = int(d.sum())
            if n:
                m = int((x.to(torch.int64) - y.to(torch.int64)).abs().max())
                return ("int_exact", str(n) + "/" + str(x.numel()) + " differ max_abs=" + str(m))
            continue
        xf = x.double()
        yf = y.double()
        fa = torch.isfinite(xf)
        fb = torch.isfinite(yf)
        if not torch.equal(fa, fb):
            n = int((fa ^ fb).sum())
            return ("finiteness", "finite/nonfinite class differs at " + str(n) + " elements")
        diff = torch.where(fa & fb, (xf - yf).abs(), torch.zeros_like(xf))
        den = yf.abs().clamp_min(1e-12)
        ma = float(diff.max())
        mr = float((diff / den).max())
        if ma > 1e-3 and mr > 1e-2:
            return ("float_large", "max_abs=%.4g max_rel=%.4g" % (ma, mr))
    return (None, "same")


print("probe=compile %s %s backend=" % (PROBE_NAME, dev) + backend, flush=True)
_eager = _run(f)
try:
    _cf = torch.compile(f, backend=backend, fullgraph=FULLGRAPH, dynamic=DYNAMIC)
except Exception as _e:
    print("COMPILE_UNSUPPORTED kind=construct detail=" + type(_e).__name__ + ":" + str(_e)[:140], flush=True)
    raise SystemExit(0)
_comp = _run(_cf)

if isinstance(_comp, Exception) and not isinstance(_eager, Exception) and _is_infra(_comp):
    print("COMPILE_UNSUPPORTED kind=backend detail=" + type(_comp).__name__ + ":" + str(_comp)[:160], flush=True)
    raise SystemExit(0)

_kind, _detail = _diff(_eager, _comp)
if _kind == "exception_type":
    print("COMPILE_EXC_TYPE detail=" + str(_detail)[:180], flush=True)
    raise SystemExit(0)
if _kind:
    print("COMPILE_DIVERGENCE kind=" + str(_kind) + " detail=" + str(_detail)[:220], flush=True)
    raise SystemExit(7)
print("COMPILE_OK", flush=True)
'''


def _body(name: str, dev: str, fn_src: str, inp_src: str, dynamic: bool, fullgraph: bool) -> str:
    if dev == "cuda":
        guard = (
            'dev = "cuda"\n'
            "if not torch.cuda.is_available():\n"
            '    print("SKIP: no cuda", flush=True); raise SystemExit(0)'
        )
    else:
        guard = 'dev = "cpu"'
    header = (
        guard
        + "\n"
        + f'PROBE_NAME = "{name}"\n'
        + f"FULLGRAPH = {bool(fullgraph)}\n"
        + f"DYNAMIC = {bool(dynamic)}\n"
        + "torch.manual_seed(0)\n"
    )
    return header + fn_src + "\n" + inp_src + "\n" + _EPILOGUE


def compile_probes(devices: tuple[str, ...] = ("cpu", "cuda")) -> list[dict[str, Any]]:
    """Return compile-differential child probes as dicts the fuzzer can run."""
    out: list[dict[str, Any]] = []
    for name, fn_src, inp_src, case_devs, dynamic, fullgraph in _CASES:
        for dev in case_devs:
            if dev not in devices:
                continue
            pname = f"compile_{name}_{dev}"
            out.append(
                {
                    "name": pname,
                    "body": _body(name, dev, fn_src, inp_src, dynamic, fullgraph),
                    "category": "compile_diff",
                    "timeout": COMPILE_TIMEOUT,
                }
            )
    return out


def catalog_meta() -> list[dict[str, Any]]:
    """One entry per compile case (device-agnostic) for reporting / Gemma seeds."""
    return [
        {
            "name": name,
            "fn_src": fn_src,
            "inp_src": inp_src,
            "devices": list(case_devs),
            "dynamic": dynamic,
            "fullgraph": fullgraph,
        }
        for name, fn_src, inp_src, case_devs, dynamic, fullgraph in _CASES
    ]


def standalone_source(name: str, dev: str = "cuda", backend: str = "inductor") -> str:
    """A fully self-contained eager-vs-compiled probe for one case (needs only torch).

    Used as the Gemma 4 SFT generation target: it teaches the model to write a
    complete differential test, not just to memorize a found crash.
    """
    case = next(c for c in _CASES if c[0] == name)
    _, fn_src, inp_src, _devs, dynamic, fullgraph = case
    head = (
        "import os\n"
        f'os.environ.setdefault("CALM_COMPILE_BACKEND", "{backend}")\n'
        "import torch\n"
    )
    return head + _body(name, dev, fn_src, inp_src, dynamic, fullgraph)


In [ ]:
%%writefile calm_common_gpu_fuzzer.py
#!/usr/bin/env python3
"""
CALM common-GPU fuzzer.

This is the "use it for real" wrapper around calm_hunt_v2_common_gpu.py.
It keeps the proven repro probes, then expands into a budgeted catalog over
software-layer surfaces that should reproduce across GPUs:

  * torch.linalg domain validation and backend error paths
  * sparse tensor invariant gaps
  * indexing/embedding invalid input handling
  * storage/lifetime metadata hazards
  * conservative CPU<->CUDA exact differential controls

The fuzzer is deliberately conservative. Normal Python RuntimeError on an
invalid input is not counted as a bug. Actionable means one of:
  * native crash / hard exit / backend abort
  * PyTorch INTERNAL ASSERT / "please report a bug"
  * CUDA device-side assert instead of clean validation
  * silent materialization of a value that check_invariants=True rejects
  * conservative CPU<->CUDA value divergence on valid exact cases
"""

from __future__ import annotations

import argparse
import hashlib
import json
import random
import sys
import time
from pathlib import Path
from typing import Any

import os

import torch

import calm_hunt_v2_common_gpu as v2
import calm_compile_diff_probes as cdp

try:
    sys.stdout.reconfigure(encoding="utf-8")
except Exception:
    pass

ROOT = Path(__file__).resolve().parent
OUT_JSON = ROOT / "calm_common_gpu_fuzzer_findings.json"
OUT_MD = ROOT / "calm_common_gpu_fuzzer_summary.md"
OUT_GEMMA_ALL = ROOT / "calm_common_gpu_gemma4_sft_all.jsonl"
OUT_GEMMA_NOVEL = ROOT / "calm_common_gpu_gemma4_sft_novel.jsonl"
OUT_GEMMA_SEEDS = ROOT / "calm_common_gpu_gemma4_seeds.jsonl"
REPRO_DIR = ROOT / "calm_common_gpu_repros"


KNOWN_DUP_HINTS = [
    {
        "key": "sgelsy",
        "level": "likely_duplicate",
        "why": "quick GitHub search found existing torch.linalg.lstsq + NaN + SGELSY/INTERNAL ASSERT reports",
        "refs": [
            "https://github.com/pytorch/pytorch/issues/138451",
            "https://github.com/pytorch/pytorch/issues/125892",
            "https://github.com/pytorch/pytorch/issues/132693",
        ],
    },
    {
        "key": "srcindex < srcselectdimsize",
        "level": "likely_duplicate",
        "why": "CUDA indexing/embedding OOB device-side assert is a known issue family",
        "refs": [
            "https://github.com/pytorch/pytorch/issues/71572",
            "https://github.com/pytorch/pytorch/issues/121493",
            "https://github.com/pytorch/pytorch/issues/151918",
        ],
    },
    {
        "key": "device-side assert",
        "level": "known_related",
        "why": "generic CUDA assert; keep only if the API family is not plain indexing/embedding",
        "refs": ["https://github.com/pytorch/pytorch/issues/71572"],
    },
    {
        "key": "silent_invalid_sparse",
        "level": "known_related",
        "why": "invalid sparse tensors with check_invariants=False have related PyTorch reports; exact variant still worth minimizing",
        "refs": ["https://github.com/pytorch/pytorch/issues/124168"],
    },
    {
        "key": "check_invariants_true_raises",
        "level": "known_related",
        "why": "sparse invariant bypass family; train Gemma to generate validated oracle, not just crash repro",
        "refs": ["https://github.com/pytorch/pytorch/issues/124168"],
    },
    {
        "key": "sgebal",
        "level": "unknown_after_quick_search",
        "why": "quick search found lstsq/SGELSY duplicates but not this exact eig/eigvals SGEBAL NaN/Inf abort pattern",
        "refs": [],
    },
    {
        "key": "resize_zero",
        "level": "unknown_after_quick_search",
        "why": "storage resize-to-zero hazards are used internally by FSDP but quick search did not find this minimal crash signature",
        "refs": ["https://github.com/pytorch/pytorch/issues/114299"],
    },
    {
        "key": "setstorage",
        "level": "unknown_after_quick_search",
        "why": "storage metadata hazard; quick search did not find an exact minimal clone/add/sum crash duplicate",
        "refs": ["https://github.com/pytorch/pytorch/issues/114299"],
    },
]


# Honest "tra truoc" screen for the eager-vs-compiled surface. These reference
# reported Inductor silent-correctness families so a hit on a known pattern is
# labelled as such instead of being passed off as novel. The structural probes
# (x-x on NaN, where() NaN dead branch, std/var of a constant, inf-inf, ...) are
# not tied to one issue; they are marked not_seen and kept as novel candidates.
COMPILE_TROUBLESHOOTING = "https://docs.pytorch.org/docs/stable/torch.compiler_troubleshooting.html"
COMPILE_DUP_TABLE = [
    ("amax", "likely_duplicate", "multiple amax/amin reductions on one tensor hit Inductor reuse_partial; reported", ["https://github.com/pytorch/pytorch/issues/179577"]),
    ("amin", "likely_duplicate", "multiple amax/amin reductions on one tensor hit Inductor reuse_partial; reported", ["https://github.com/pytorch/pytorch/issues/179577"]),
    ("transpose", "likely_duplicate", "Inductor silent correctness when fusing transpose into other ops; reported", ["https://github.com/pytorch/pytorch/issues/146416"]),
    ("addcdiv", "likely_duplicate", "addcdiv(x,x,clamp(x,min=1e-6)) wrong under inductor; reported", ["https://github.com/pytorch/pytorch/issues/181694"]),
    ("layernorm", "known_related", "LayerNorm/norm numerical error under inductor is a known family", ["https://github.com/pytorch/pytorch/issues/143182"]),
    ("index_add", "known_related", "index_add_ mismatch between inductor and eager; reported", ["https://github.com/pytorch/pytorch/issues/111133"]),
    ("softmax_bigdim", "known_related", "online-softmax disabled on reduction split; large-dim softmax is a known area", ["https://github.com/pytorch/pytorch/issues/157975"]),
    ("groupnorm", "known_related", "norm-family numerics under inductor", ["https://github.com/pytorch/pytorch/issues/143182"]),
    ("matmul_layernorm", "known_related", "norm-family numerics under inductor", ["https://github.com/pytorch/pytorch/issues/143182"]),
]


def compile_dup_hint(text: str) -> dict[str, Any]:
    for key, level, why, refs in COMPILE_DUP_TABLE:
        if key in text:
            return {"level": level, "why": why, "refs": refs}
    # structural / extreme-input probe: no specific reported issue
    return {
        "level": "not_seen_in_quick_search",
        "why": "structural eager-vs-compiled divergence; no specific reported issue from the quick scan, retest on nightly before claiming novel",
        "refs": [COMPILE_TROUBLESHOOTING],
    }


def q(s: str) -> str:
    return s.replace("\\", "\\\\").replace('"', '\\"')


def linalg_body(op: str, dev: str, variant: str, shape: str = "2x2") -> str:
    """Return a child script body for a linalg op."""
    device_line = (
        'dev = "cuda"\nif not torch.cuda.is_available():\n'
        '    print("SKIP: no cuda", flush=True); raise SystemExit(0)'
        if dev == "cuda"
        else 'dev = "cpu"'
    )
    if shape == "2x2":
        base = 'A = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device=dev)'
        b = 'B = torch.ones(2, 1, device=dev)'
    elif shape == "3x2":
        base = 'A = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]], device=dev)'
        b = 'B = torch.ones(3, 1, device=dev)'
    else:
        base = 'A = torch.eye(4, device=dev)'
        b = 'B = torch.ones(4, 1, device=dev)'

    if variant == "offdiag_nan":
        mutate = 'A = A.clone(); A[0, 1] = float("nan"); A[1, 0] = float("nan")'
    elif variant == "diag_nan":
        mutate = 'A = A.clone(); A[0, 0] = float("nan")'
    elif variant == "offdiag_inf":
        mutate = 'A = A.clone(); A[0, 1] = float("inf"); A[1, 0] = float("-inf")'
    elif variant == "singular":
        mutate = 'A = A.clone(); A[-1] = A[0]'
    elif variant == "zero":
        mutate = 'A = torch.zeros_like(A)'
    else:
        mutate = 'A = A.clone()'

    call_map = {
        "eigvals": "Y = torch.linalg.eigvals(A)",
        "eig": "Y = torch.linalg.eig(A).eigenvalues",
        "lstsq": "Y = torch.linalg.lstsq(A, B).solution",
        "solve": "Y = torch.linalg.solve(A, B)",
        "inv": "Y = torch.linalg.inv(A)",
        "det": "Y = torch.linalg.det(A)",
        "slogdet": "Y = torch.linalg.slogdet(A)",
        "svdvals": "Y = torch.linalg.svdvals(A)",
        "pinv": "Y = torch.linalg.pinv(A)",
        "matrix_rank": "Y = torch.linalg.matrix_rank(A)",
        "cond": "Y = torch.linalg.cond(A)",
        "cholesky": "Y = torch.linalg.cholesky(A)",
    }
    call = call_map[op]
    return f"""
{device_line}
print("probe=linalg {op} {variant} {dev} {shape}", flush=True)
{base}
{b}
{mutate}
{call}
if dev == "cuda":
    torch.cuda.synchronize()
print(Y, flush=True)
"""


def sparse_body(kind: str, dev: str) -> str:
    device_line = (
        'dev = "cuda"\nif not torch.cuda.is_available():\n'
        '    print("SKIP: no cuda", flush=True); raise SystemExit(0)'
        if dev == "cuda"
        else 'dev = "cpu"'
    )
    if kind == "coo_neg_to_dense":
        build = """
idx = torch.tensor([[-1, 0], [0, 1]], device=dev)
val = torch.ones(2, device=dev)
x = torch.sparse_coo_tensor(idx, val, (2, 2), check_invariants=False)
print(x.to_dense(), flush=True)
if dev == "cpu":
    try:
        torch.sparse_coo_tensor(idx.cpu(), val.cpu(), (2, 2), check_invariants=True)
    except Exception as e:
        print("check_invariants_true_raises=", type(e).__name__, str(e)[:100], flush=True)
    print("SILENT_INVALID_SPARSE", flush=True)
"""
    elif kind == "coo_oob_to_dense":
        build = """
idx = torch.tensor([[2, 0], [0, 1]], device=dev)
val = torch.ones(2, device=dev)
x = torch.sparse_coo_tensor(idx, val, (2, 2), check_invariants=False)
print(x.to_dense(), flush=True)
if dev == "cpu":
    try:
        torch.sparse_coo_tensor(idx.cpu(), val.cpu(), (2, 2), check_invariants=True)
    except Exception as e:
        print("check_invariants_true_raises=", type(e).__name__, str(e)[:100], flush=True)
    print("SILENT_INVALID_SPARSE", flush=True)
"""
    elif kind == "csr_bad_crow_to_dense":
        build = """
crow = torch.tensor([0, 2, 1], device=dev)
col = torch.tensor([0, 1], device=dev)
val = torch.ones(2, device=dev)
x = torch.sparse_csr_tensor(crow, col, val, size=(2, 2), check_invariants=False)
print(x.to_dense(), flush=True)
if dev == "cpu":
    try:
        torch.sparse_csr_tensor(crow.cpu(), col.cpu(), val.cpu(), size=(2, 2), check_invariants=True)
    except Exception as e:
        print("check_invariants_true_raises=", type(e).__name__, str(e)[:100], flush=True)
    print("SILENT_INVALID_SPARSE", flush=True)
"""
    elif kind == "csr_oob_col_to_dense":
        build = """
crow = torch.tensor([0, 1, 2], device=dev)
col = torch.tensor([0, 2], device=dev)
val = torch.ones(2, device=dev)
x = torch.sparse_csr_tensor(crow, col, val, size=(2, 2), check_invariants=False)
print(x.to_dense(), flush=True)
if dev == "cpu":
    try:
        torch.sparse_csr_tensor(crow.cpu(), col.cpu(), val.cpu(), size=(2, 2), check_invariants=True)
    except Exception as e:
        print("check_invariants_true_raises=", type(e).__name__, str(e)[:100], flush=True)
    print("SILENT_INVALID_SPARSE", flush=True)
"""
    else:
        build = 'print("unknown sparse kind", flush=True)'
    return f"""
{device_line}
print("probe=sparse {kind} {dev}", flush=True)
{build}
if dev == "cuda":
    torch.cuda.synchronize()
"""


def index_body(kind: str, dev: str) -> str:
    device_line = (
        'dev = "cuda"\nif not torch.cuda.is_available():\n'
        '    print("SKIP: no cuda", flush=True); raise SystemExit(0)'
        if dev == "cuda"
        else 'dev = "cpu"'
    )
    if kind == "embedding_oob":
        body = """
w = torch.randn(4, 3, device=dev)
idx = torch.tensor([0, 1, 4], device=dev, dtype=torch.long)
print(torch.nn.functional.embedding(idx, w), flush=True)
"""
    elif kind == "gather_oob":
        body = """
x = torch.arange(8, device=dev).reshape(2, 4)
idx = torch.tensor([[0, 1, 4, 2], [0, 1, 2, 3]], device=dev)
print(torch.gather(x, 1, idx), flush=True)
"""
    elif kind == "index_select_oob":
        body = """
x = torch.arange(8, device=dev).reshape(2, 4)
idx = torch.tensor([0, 2], device=dev)
print(torch.index_select(x, 0, idx), flush=True)
"""
    else:
        body = 'print("unknown index kind", flush=True)'
    return f"""
{device_line}
print("probe=index {kind} {dev}", flush=True)
{body}
if dev == "cuda":
    torch.cuda.synchronize()
"""


def storage_body(kind: str, dev: str) -> str:
    device_line = (
        'dev = "cuda"\nif not torch.cuda.is_available():\n'
        '    print("SKIP: no cuda", flush=True); raise SystemExit(0)'
        if dev == "cuda"
        else 'dev = "cpu"'
    )
    op = {
        "clone": "print(x.clone(), flush=True)",
        "add": "print(x + 1, flush=True)",
        "repr": "print(repr(x), flush=True)",
        "sum": "print(x.sum(), flush=True)",
    }[kind]
    return f"""
{device_line}
print("probe=storage resize_zero {kind} {dev}", flush=True)
x = torch.arange(4, device=dev)
if dev == "cuda":
    torch.cuda.synchronize()
s = x.untyped_storage()
print("before", x, flush=True)
s.resize_(0)
if dev == "cuda":
    torch.cuda.synchronize()
print("after resize", flush=True)
{op}
if dev == "cuda":
    torch.cuda.synchronize()
"""


def digest_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()[:12]


def row_text(row: dict[str, Any]) -> str:
    return (
        str(row.get("name", ""))
        + "\n"
        + str(row.get("category", ""))
        + "\n"
        + str(row.get("stdout_tail", ""))
        + "\n"
        + str(row.get("stderr_tail", ""))
    ).lower()


def dup_hint_for(row: dict[str, Any]) -> dict[str, Any]:
    text = row_text(row)
    if row.get("category") == "compile_diff":
        return compile_dup_hint(text)
    if "sgelsy" in text:
        hint = KNOWN_DUP_HINTS[0]
        return {"level": hint["level"], "why": hint["why"], "refs": hint["refs"]}
    if "sgebal" in text:
        hint = next(h for h in KNOWN_DUP_HINTS if h["key"] == "sgebal")
        return {"level": hint["level"], "why": hint["why"], "refs": hint["refs"]}
    if "resize_zero" in text or "resize zero" in text or "untyped_storage" in text or "storage resize" in text:
        hint = next(h for h in KNOWN_DUP_HINTS if h["key"] == "resize_zero")
        return {"level": hint["level"], "why": hint["why"], "refs": hint["refs"]}
    if "silent_invalid_sparse" in text or "check_invariants_true_raises" in text:
        hint = next(h for h in KNOWN_DUP_HINTS if h["key"] == "silent_invalid_sparse")
        return {"level": hint["level"], "why": hint["why"], "refs": hint["refs"]}
    if "srcindex < srcselectdimsize" in text:
        hint = next(h for h in KNOWN_DUP_HINTS if h["key"] == "srcindex < srcselectdimsize")
        return {"level": hint["level"], "why": hint["why"], "refs": hint["refs"]}
    for hint in KNOWN_DUP_HINTS:
        if hint["key"] in text:
            if hint["key"] == "device-side assert" and row.get("category") == "indexing":
                continue
            return {
                "level": hint["level"],
                "why": hint["why"],
                "refs": hint["refs"],
            }
    return {
        "level": "not_seen_in_quick_search",
        "why": "no matching built-in duplicate signature from the quick PyTorch issue scan",
        "refs": [],
    }


def catalog(seed: int, compile_devices: tuple[str, ...] = ("cpu", "cuda")) -> list[dict[str, Any]]:
    rng = random.Random(seed)
    core: list[dict[str, Any]] = []
    priority: list[dict[str, Any]] = []
    linalg_rows: list[dict[str, Any]] = []

    # Proven v2 repros first.
    for name, body in v2.child_probes():
        core.append({"name": "v2_" + name, "body": body, "category": "v2_core"})

    # Eager-vs-compiled equivalence differential. This is the surface that needs
    # a real accelerator + Inductor (Colab T4 / Blackwell); it sits right after
    # the proven core so it always runs inside the budget.
    compile_rows = cdp.compile_probes(devices=compile_devices)
    rng.shuffle(compile_rows)

    for dev in ("cpu", "cuda"):
        for op in ("eigvals", "eig", "lstsq", "solve", "inv", "det", "slogdet", "svdvals", "pinv", "matrix_rank", "cond", "cholesky"):
            variants = ("offdiag_nan", "diag_nan", "offdiag_inf", "singular", "zero")
            for variant in variants:
                shapes = ("2x2", "3x2") if op == "lstsq" else ("2x2",)
                for shape in shapes:
                    name = f"lin_{op}_{variant}_{shape}_{dev}"
                    linalg_rows.append({"name": name, "body": linalg_body(op, dev, variant, shape), "category": "linalg"})

    for dev in ("cpu", "cuda"):
        for kind in ("coo_neg_to_dense", "coo_oob_to_dense", "csr_bad_crow_to_dense", "csr_oob_col_to_dense"):
            priority.append({"name": f"sparse_{kind}_{dev}", "body": sparse_body(kind, dev), "category": "sparse"})
        for kind in ("embedding_oob", "gather_oob", "index_select_oob"):
            priority.append({"name": f"index_{kind}_{dev}", "body": index_body(kind, dev), "category": "indexing"})
        for kind in ("clone", "add", "repr", "sum"):
            priority.append({"name": f"storage_resize_zero_{kind}_{dev}", "body": storage_body(kind, dev), "category": "storage"})

    rng.shuffle(priority)
    rng.shuffle(linalg_rows)
    return core + compile_rows + priority + linalg_rows


def refine(row: dict[str, Any], category: str) -> dict[str, Any]:
    row["category"] = category
    text = ((row.get("stdout_tail") or "") + "\n" + (row.get("stderr_tail") or "")).lower()

    if category == "compile_diff":
        if "compile_divergence" in text:
            kind = "value"
            for tok in text.splitlines():
                if "compile_divergence" in tok and "kind=" in tok:
                    kind = tok.split("kind=", 1)[1].split()[0]
                    break
            row["status"] = "compile_divergence"
            row["verdict"] = f"Tier A: eager vs compiled silent divergence ({kind})"
        elif "compile_unsupported" in text or "compile_exc_type" in text:
            row["status"] = "expected_raise"
            row["verdict"] = "Compile backend unsupported / benign exception re-wrap (negative control)"
        elif row.get("status") in {"internal_assert", "cuda_assert", "backend_abort", "native_crash"} or row.get("returncode") == 3221225477:
            row["verdict"] = "Tier A: crash/assert inside torch.compile codegen"
        elif row.get("status") == "ok" and "compile_ok" not in text and "skip:" not in text:
            # non-zero exit with no recognized marker: keep whatever v2 said
            pass
        row["dup_hint"] = dup_hint_for(row)
        return row

    strong = any(
        k in text
        for k in (
            "internal assert",
            "please report a bug",
            "device-side assert",
            "illegal memory access",
            "intel onemkl error",
            "sgebal",
            "sgelsy",
        )
    )
    if row.get("returncode") == 3221225477:
        strong = True
    if row["status"] == "runtime_error":
        if not strong and category in {"linalg", "indexing"}:
            row["status"] = "expected_raise"
            row["verdict"] = "Clean RuntimeError on invalid/domain input; not counted"
        elif not strong and category == "sparse":
            row["status"] = "sparse_runtime"
            row["verdict"] = "Sparse invalid input reached runtime error; triage manually"
        elif not strong and category == "storage":
            row["status"] = "storage_runtime"
            row["verdict"] = "Storage metadata hazard reached runtime error"
    if row["status"] == "hard_exit" and row.get("returncode") == 1 and not strong:
        if category in {"linalg", "indexing"}:
            row["status"] = "expected_raise"
            row["verdict"] = "Clean domain/index exception; not counted"
        elif category == "sparse":
            row["status"] = "expected_raise"
            row["verdict"] = "Clean sparse/index exception; not counted"
    if row["status"] == "hard_exit" and "0xc0000005" not in text and row.get("returncode") == 3221225477:
        row["verdict"] = "Tier A: Windows access violation / native crash"
    row["dup_hint"] = dup_hint_for(row)
    return row


def actionable(row: dict[str, Any]) -> bool:
    return row.get("status") not in {"ok", "skip", "expected_raise", None}


def sig(row: dict[str, Any]) -> tuple[str, str, str]:
    text = ((row.get("stdout_tail") or "") + "\n" + (row.get("stderr_tail") or "")).lower()
    if row.get("category") == "compile_diff":
        fam = row.get("name", "")
        for suf in ("_cpu", "_cuda"):
            if fam.endswith(suf):
                fam = fam[: -len(suf)]
        kind = ""
        for tok in text.splitlines():
            if "kind=" in tok:
                kind = tok.split("kind=", 1)[1].split()[0]
                break
        return (row.get("category", ""), row.get("status", ""), fam + ":" + kind)
    if "sgebal" in text:
        s = "sgebal"
    elif "sgelsy" in text:
        s = "sgelsy"
    elif "device-side assert" in text:
        s = "device-side assert"
    elif "setstorage" in text:
        s = "setstorage"
    elif "0xc0000005" in text or row.get("returncode") == 3221225477:
        s = "access violation"
    else:
        s = text[:160]
    return (row.get("category", ""), row.get("status", ""), s)


def write_summary(payload: dict[str, Any]) -> None:
    hits = [r for r in payload["results"] if actionable(r)]
    unique = {}
    for r in hits:
        unique.setdefault(sig(r), r)
    dup_counts: dict[str, int] = {}
    for row in unique.values():
        level = row.get("dup_hint", {}).get("level", "unknown")
        dup_counts[level] = dup_counts.get(level, 0) + 1
    lines = [
        "# CALM common-GPU fuzzer summary",
        "",
        f"- torch: `{payload['torch']}`",
        f"- cuda: `{payload['cuda']}`",
        f"- gpu: `{payload.get('gpu_name') or 'none'}`",
        f"- executed child probes: **{payload['summary']['executed_child']}**",
        f"- actionable hits: **{len(hits)}**",
        f"- unique signatures: **{len(unique)}**",
        f"- novelty/dup screen: `{dup_counts}`",
        f"- exact CPU/CUDA controls: `{payload['summary']['device_diff_hits']}` conservative divergences",
        f"- compile backend: `{payload.get('compile_backend')}` on devices `{payload.get('compile_devices')}`",
        f"- compile-diff verdicts: `{payload['summary'].get('compile_diff', {})}`",
        f"- compile preflight: `{payload['compile_preflight']['detail']}`",
        f"- Gemma SFT all rows: `{OUT_GEMMA_ALL.name}`",
        f"- Gemma SFT novel rows: `{OUT_GEMMA_NOVEL.name}`",
        f"- Gemma compile-diff seed rows: `{OUT_GEMMA_SEEDS.name}` ({payload['summary'].get('gemma_seed_rows', 0)} method seeds)",
        "",
        "## Unique actionable signatures",
        "",
    ]
    for row in unique.values():
        lines.append(f"- **{row['name']}** [{row['category']}]: `{row['status']}` - {row.get('verdict', '')}")
        if row.get("repro"):
            lines.append(f"  - repro: `{row['repro']}`")
        if row.get("dup_hint"):
            dh = row["dup_hint"]
            refs = ", ".join(dh.get("refs", [])[:3]) or "none"
            lines.append(f"  - dup_screen: `{dh.get('level')}` - {dh.get('why')} | refs: {refs}")
        sample = ((row.get("stdout_tail") or "") + "\n" + (row.get("stderr_tail") or "")).strip().splitlines()[:4]
        if sample:
            lines.append("  - signal: `" + " | ".join(x[:150] for x in sample) + "`")
    OUT_MD.write_text("\n".join(lines) + "\n", encoding="utf-8")


def axis_for(row: dict[str, Any]) -> str:
    cat = row.get("category", "")
    status = row.get("status", "")
    name = row.get("name", "")
    low_name = name.lower()
    if "compile" in name:
        return "compile_drift"
    if cat == "cpu_cuda_exact":
        return "device_divergence"
    if cat == "linalg" or "eig" in low_name or "lstsq" in low_name or "linalg" in low_name:
        return "metamorphic_axiom"
    if cat == "sparse" or "sparse" in low_name or status in {"contract_silent", "sparse_runtime"}:
        return "boundary_contract"
    if cat == "storage" or "storage" in low_name or "resize_zero" in low_name or "resize zero" in low_name:
        return "boundary_contract"
    if cat == "indexing" or "embedding" in low_name or "gather" in low_name or "index_select" in low_name:
        return "boundary_contract"
    return "op_core"


def focus_for(axis: str) -> str:
    return {
        "device_divergence": "Compare CPU and CUDA behavior with conservative exact oracles.",
        "compile_drift": "Contrast eager execution with compiled/graph execution of the same computation.",
        "metamorphic_axiom": "Probe mathematical boundary invariants such as NaN/Inf handling and backend contracts.",
        "boundary_contract": "Stress API boundary contracts and require clean validation instead of native/device failure.",
        "op_core": "Keep the operator chain short and assert on concrete behavior.",
    }.get(axis, "Keep the reproducer minimal and deterministic.")


def api_hint_for(row: dict[str, Any]) -> list[str]:
    name = row.get("name", "")
    cat = row.get("category", "")
    low_name = name.lower()
    if cat == "compile_diff" or low_name.startswith("compile_"):
        return ["torch.compile", "torch._inductor", "torch._dynamo"]
    if "storage" in low_name or "resize_zero" in low_name or "resize zero" in low_name:
        return ["Tensor.untyped_storage", "UntypedStorage.resize_", "Tensor.clone", "Tensor.sum"]
    if "sparse" in low_name or cat == "sparse":
        return ["torch.sparse_coo_tensor", "torch.sparse_csr_tensor", "Tensor.to_dense"]
    if "embedding" in low_name or "gather" in low_name or "index_select" in low_name or cat == "indexing":
        return ["torch.nn.functional.embedding", "torch.gather", "torch.index_select"]
    if "eig" in name:
        return ["torch.linalg.eig", "torch.linalg.eigvals"]
    if "lstsq" in name:
        return ["torch.linalg.lstsq"]
    return ["torch"]


def repro_code_for(row: dict[str, Any]) -> str:
    path = row.get("repro")
    if path:
        p = Path(path)
        if p.exists():
            return p.read_text(encoding="utf-8", errors="replace")
    return (
        "import torch\n\n"
        "# Re-run the fuzzer to regenerate the minimal repro for this row.\n"
        f"print({row.get('name', 'unknown')!r})\n"
    )


def gemma_row(row: dict[str, Any], novelty: str) -> dict[str, Any]:
    axis = axis_for(row)
    code = repro_code_for(row)
    dh = row.get("dup_hint", {})
    rid = f"calm-common-gpu:{row.get('name')}:{digest_text(code)}"
    instruction = (
        "Generate a minimal, self-contained PyTorch reliability fuzzing seed. "
        "Prefer a child-process-safe repro with a concrete oracle and no external data."
    )
    inp = "\n".join(
        [
            "Framework: pytorch",
            f"Focus: {focus_for(axis)}",
            f"Observed status: {row.get('status')}",
            f"Surface: {row.get('category')} / {row.get('name')}",
            f"Novelty screen: {dh.get('level', 'unknown')} - {dh.get('why', '')}",
            "APIs: " + ", ".join(api_hint_for(row)),
            "Requirement: distinguish clean RuntimeError/IndexError from native crash, INTERNAL ASSERT, CUDA assert, or silent invariant gap.",
        ]
    )
    text = (
        "<start_of_turn>user\n"
        + instruction
        + "\n\n"
        + inp
        + "<end_of_turn>\n<start_of_turn>model\n```python\n"
        + code.rstrip()
        + "\n```<end_of_turn>\n"
    )
    return {
        "id": rid,
        "framework": "pytorch",
        "axis": axis,
        "repo": "pytorch/pytorch",
        "issue_number": None,
        "url": None,
        "quality_score": 100 if novelty == "novel" else 72,
        "calm_score": 5.0 if novelty == "novel" else 3.5,
        "n_code_lines": len(code.splitlines()),
        "instruction": instruction,
        "input": inp,
        "output": code,
        "text": text,
        "source": "calm_common_gpu_fuzzer",
        "run_status": row.get("status"),
        "novelty": novelty,
        "dup_hint": dh,
        "meta": {
            "name": row.get("name"),
            "category": row.get("category"),
            "verdict": row.get("verdict"),
            "repro": row.get("repro"),
            "signal_sha1": digest_text(row_text(row)),
        },
    }


def write_gemma_rows(payload: dict[str, Any]) -> None:
    hits = [r for r in payload["results"] if actionable(r)]
    unique: dict[tuple[str, str, str], dict[str, Any]] = {}
    for row in hits:
        unique.setdefault(sig(row), row)

    all_rows = []
    novel_rows = []
    for row in unique.values():
        level = row.get("dup_hint", {}).get("level", "unknown")
        novelty = "novel" if level in {"not_seen_in_quick_search", "unknown_after_quick_search"} else "known_related"
        out_row = gemma_row(row, novelty)
        all_rows.append(out_row)
        if novelty == "novel":
            novel_rows.append(out_row)

    OUT_GEMMA_ALL.write_text(
        "\n".join(json.dumps(r, ensure_ascii=False) for r in all_rows) + ("\n" if all_rows else ""),
        encoding="utf-8",
    )
    OUT_GEMMA_NOVEL.write_text(
        "\n".join(json.dumps(r, ensure_ascii=False) for r in novel_rows) + ("\n" if novel_rows else ""),
        encoding="utf-8",
    )


def write_gemma_seed_rows(backend: str) -> int:
    """Export the compile-diff catalog as Gemma 4 generation seeds.

    These rows teach the model the eager-vs-compiled differential *method* and
    the high-signal patterns, independent of whether a divergence was found on
    this build. Output is a complete, self-contained probe (the generation
    target), so the model learns to write differential tests, not memorize a
    single crash. This is how the compile surface serves Gemma even when a
    stable build yields no divergence.
    """
    rows = []
    for meta in cdp.catalog_meta():
        name = meta["name"]
        code = cdp.standalone_source(name, dev="cuda", backend=backend)
        dh = compile_dup_hint(name.lower())
        instruction = (
            "Generate a self-contained PyTorch eager-vs-compiled differential fuzzing probe. "
            "Run the same function eagerly and under torch.compile on the same device and dtype, "
            "then compare with a conservative oracle that flags only finiteness-class, shape/dtype, "
            "or large numeric divergence, and treats a compile backend limitation as a negative control."
        )
        inp = "\n".join(
            [
                "Framework: pytorch",
                "Focus: Contrast eager execution with torch.compile (Inductor) of the same computation.",
                f"Pattern: {name}",
                f"Novelty screen: {dh.get('level')} - {dh.get('why')}",
                "APIs: torch.compile, torch._inductor, torch._dynamo",
                "Requirement: child-process safe; print COMPILE_DIVERGENCE/COMPILE_OK/COMPILE_UNSUPPORTED; no external data.",
            ]
        )
        text = (
            "<start_of_turn>user\n"
            + instruction
            + "\n\n"
            + inp
            + "<end_of_turn>\n<start_of_turn>model\n```python\n"
            + code.rstrip()
            + "\n```<end_of_turn>\n"
        )
        rows.append(
            {
                "id": f"calm-compile-seed:{name}:{digest_text(code)}",
                "framework": "pytorch",
                "axis": "compile_drift",
                "repo": "pytorch/pytorch",
                "issue_number": None,
                "url": (dh.get("refs") or [None])[0],
                "quality_score": 95,
                "calm_score": 4.5,
                "n_code_lines": len(code.splitlines()),
                "instruction": instruction,
                "input": inp,
                "output": code,
                "text": text,
                "source": "calm_compile_diff_catalog",
                "novelty": "method_seed",
                "dup_hint": dh,
                "meta": {"name": name, "category": "compile_diff"},
            }
        )
    OUT_GEMMA_SEEDS.write_text(
        "\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + ("\n" if rows else ""),
        encoding="utf-8",
    )
    return len(rows)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--seconds", type=float, default=180.0)
    ap.add_argument("--max-child", type=int, default=80)
    ap.add_argument("--timeout", type=float, default=18.0)
    ap.add_argument("--seed", type=int, default=20260611)
    ap.add_argument("--compile-backend", default="inductor",
                    help="torch.compile backend for the eager-vs-compiled surface (inductor on T4/Blackwell; aot_eager to self-test without Triton/cl)")
    ap.add_argument("--compile-devices", default="cpu,cuda",
                    help="comma list of devices for compile-diff probes")
    args = ap.parse_args()

    os.environ["CALM_COMPILE_BACKEND"] = args.compile_backend
    compile_devices = tuple(d.strip() for d in args.compile_devices.split(",") if d.strip())

    v2.REPRO_DIR = REPRO_DIR
    print("=" * 80)
    print(f"CALM common-GPU fuzzer | torch={torch.__version__} | cuda={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU={torch.cuda.get_device_name(0)} cap={torch.cuda.get_device_capability(0)}")
    print(f"budget={args.seconds:.0f}s max_child={args.max_child} seed={args.seed} "
          f"compile_backend={args.compile_backend} compile_devices={compile_devices}")
    print("=" * 80)

    t0 = time.time()
    results: list[dict[str, Any]] = []
    probes = catalog(args.seed, compile_devices=compile_devices)
    executed = 0
    for p in probes:
        if executed >= args.max_child or time.time() - t0 > args.seconds:
            break
        timeout = float(p.get("timeout", args.timeout))
        row = v2.run_child_probe(p["name"], p["body"], timeout_sec=timeout)
        row = refine(row, p["category"])
        results.append(row)
        executed += 1
        mark = "HIT" if actionable(row) else row["status"].upper()
        print(f"{executed:03d} {mark:14s} {p['category']:11s} {p['name'][:46]:46s} {row['status']}")

    dd = v2.run_device_differential()
    for row in dd:
        row["category"] = "cpu_cuda_exact"
    cpre = v2.compile_preflight()
    results.extend(dd)

    by_status: dict[str, int] = {}
    by_cat: dict[str, int] = {}
    novelty_counts: dict[str, int] = {}
    compile_diff: dict[str, int] = {}
    for row in results:
        by_status[row["status"]] = by_status.get(row["status"], 0) + 1
        by_cat[row.get("category", "?")] = by_cat.get(row.get("category", "?"), 0) + 1
        if row.get("category") == "compile_diff":
            compile_diff[row["status"]] = compile_diff.get(row["status"], 0) + 1
        if actionable(row):
            level = row.get("dup_hint", {}).get("level", "unknown")
            novelty_counts[level] = novelty_counts.get(level, 0) + 1
    hits = [r for r in results if actionable(r)]
    unique = {}
    for r in hits:
        unique.setdefault(sig(r), r)
    payload = {
        "tool": "calm_common_gpu_fuzzer",
        "torch": torch.__version__,
        "cuda": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "gpu_capability": torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
        "budget_sec": args.seconds,
        "seed": args.seed,
        "compile_backend": args.compile_backend,
        "compile_devices": list(compile_devices),
        "compile_preflight": cpre,
        "results": results,
        "summary": {
            "executed_child": executed,
            "total_results": len(results),
            "actionable": len(hits),
            "unique_actionable_signatures": len(unique),
            "device_diff_hits": len(dd),
            "compile_diff": compile_diff,
            "by_status": by_status,
            "by_category": by_cat,
            "by_dup_hint": novelty_counts,
            "elapsed_sec": round(time.time() - t0, 3),
        },
    }
    OUT_JSON.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    write_gemma_rows(payload)
    n_seeds = write_gemma_seed_rows(args.compile_backend)
    payload["summary"]["gemma_seed_rows"] = n_seeds
    OUT_JSON.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    write_summary(payload)
    print("-" * 80)
    print(json.dumps(payload["summary"], ensure_ascii=False, indent=2))
    print(f"-> {OUT_JSON}")
    print(f"-> {OUT_MD}")
    print(f"-> {OUT_GEMMA_SEEDS} ({n_seeds} compile seeds)")
    print(f"-> repros {REPRO_DIR}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Compile preflight

Check that the requested `torch.compile` backend actually works on this machine. On Colab T4 / Blackwell, Inductor + Triton are present and this enables the real eager-vs-compiled hunt. On a machine without a C compiler / Triton (e.g. local Windows), it transparently falls back to `aot_eager`, which still validates the differential harness end-to-end — it just will not exercise Inductor codegen, so a clean run there means "harness sound, Inductor surface deferred", not "no bugs".


In [ ]:
def _backend_ok(backend):
    try:
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        g = torch.compile(lambda x: (x - x).sum() + torch.softmax(x, dim=-1).sum(),
                          backend=backend, fullgraph=True)
        y = g(torch.randn(8, device=dev))
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        float(y)
        return True, ""
    except Exception as e:
        return False, f"{type(e).__name__}: {str(e)[:200]}"

ok, detail = _backend_ok(COMPILE_BACKEND)
if ok:
    print(f"[preflight] backend '{COMPILE_BACKEND}' works -> real eager-vs-compiled hunt enabled")
else:
    print(f"[preflight] '{COMPILE_BACKEND}' unavailable: {detail}")
    alt_ok, _ = _backend_ok("aot_eager")
    COMPILE_BACKEND = "aot_eager" if alt_ok else "eager"
    print(f"[preflight] falling back to '{COMPILE_BACKEND}' "
          f"(harness self-test; Inductor codegen surface deferred to a Triton-capable machine)")
print("COMPILE_BACKEND =", COMPILE_BACKEND, "| COMPILE_DEVICES =", COMPILE_DEVICES)


## Run fuzzer

Each hazardous probe runs in a child process, so native crashes, device-side asserts, Inductor codegen segfaults, and poisoned CUDA contexts do not kill the notebook kernel.


In [ ]:
cmd = [
    sys.executable,
    "calm_common_gpu_fuzzer.py",
    "--seconds", str(FUZZ_SECONDS),
    "--max-child", str(MAX_CHILD),
    "--timeout", str(CHILD_TIMEOUT),
    "--seed", str(SEED),
    "--compile-backend", COMPILE_BACKEND,
    "--compile-devices", COMPILE_DEVICES,
]
print(" ".join(cmd))
proc = subprocess.run(cmd, text=True)
print("returncode", proc.returncode)


## Summarize and package artifacts


In [ ]:
from collections import defaultdict

payload = json.loads(Path("calm_common_gpu_fuzzer_findings.json").read_text(encoding="utf-8"))
summary = payload["summary"]
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\ncompile backend used:", payload.get("compile_backend"), "on", payload.get("compile_devices"))
print("compile-diff verdicts:", summary.get("compile_diff", {}))
print("  COMPILE_OK            = eager and compiled agreed (good)")
print("  compile_divergence    = silent eager-vs-compiled value divergence (Tier A; triage)")
print("  expected_raise        = backend unsupported / benign exception re-wrap (negative control)")

hits = [r for r in payload["results"] if r["status"] not in {"ok", "skip", "expected_raise", None}]
by_family = defaultdict(list)
for r in hits:
    by_family[r["category"]].append(r)

print("\nActionable families:")
for fam, rows in sorted(by_family.items()):
    print(f"- {fam}: {len(rows)}")
    for r in rows[:8]:
        print("   ", r["name"], "=>", r["status"])

print("\nUnique signatures from markdown summary:")
print(Path("calm_common_gpu_fuzzer_summary.md").read_text(encoding="utf-8")[:7000])

zip_path = Path("common_gpu_fuzzer_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in [
        "calm_common_gpu_fuzzer_findings.json",
        "calm_common_gpu_fuzzer_summary.md",
        "calm_common_gpu_gemma4_sft_all.jsonl",
        "calm_common_gpu_gemma4_sft_novel.jsonl",
        "calm_common_gpu_gemma4_seeds.jsonl",
    ]:
        if Path(name).exists():
            z.write(name)
    repro_dir = Path("calm_common_gpu_repros")
    if repro_dir.exists():
        for p in sorted(repro_dir.glob("*.py")):
            z.write(p, "repros/" + p.name)
print("\nwrote", zip_path.resolve(), zip_path.stat().st_size, "bytes")


## Reading the result

Good final claims for this fuzzer:

- Report by *family*, not raw count: linalg backend/internal-assert, sparse invariant gaps, CUDA invalid-index asserts, storage metadata hazards, and the eager-vs-compiled (`compile_diff`) family.
- `compile_diff` is the surface that needed this hardware. If `compile_backend` is `inductor`, a `compile_divergence` row is a silent eager-vs-compiled disagreement on a stable build — minimize it, search existing issues (the `dup_screen` field already cites the related ones), retest on nightly, then disclose. If `compile_backend` fell back to `aot_eager`, `compile_diff` showing all `ok` means "the differential harness is sound and produced zero false positives", not "Inductor is bug-free" — rerun on a Triton-capable machine for the real Inductor pass.
- Keep `expected_raise` in the log. They are the negative controls proving the classifier is not calling every invalid input (or every compile limitation) a bug.
- `device_diff_hits=0` on the exact CPU/CUDA controls is good: the comparator did not hallucinate differences on ordinary exact operations.
- Gemma 4 exports: `calm_common_gpu_gemma4_sft_novel.jsonl` (novel/unknown findings) and `*_all.jsonl` (full triage taxonomy) for what failed; `calm_common_gpu_gemma4_seeds.jsonl` to teach the model to *generate* eager-vs-compiled differential probes regardless of whether this build diverged.
- Do not call anything a CVE from this notebook alone. The next step is retest on PyTorch nightly/main, minimize the reproducer, search existing issues, then disclose upstream.
